# 🧪 PARCIAL – UNIDAD 3
## Preparación de los Datos

**Curso:** Minería de Datos  
**Fase CRISP-DM evaluada:** Preparación de los Datos  
**Puntaje total:** 20 puntos

### Datos del grupo
- **Grupo:**  
- **Integrantes:**  
- **Proyecto:**  
- **Dataset utilizado:**  

## Objetivo
Evaluar la capacidad del estudiante para **programar, justificar y reproducir** un proceso de preparación de datos.

### Se evalúa
1. Limpieza y tratamiento de calidad.
2. Transformación y codificación.
3. Ingeniería de características.
4. Selección o reducción de variables.
5. Pipeline reproducible.
6. Dataset final preparado.

> Este parcial evalúa principalmente el **código**. Toda transformación debe mostrar evidencia antes y después.

**Grupo:** Fenix

**Integrantes:** Glender Samuel Vargas Chipana, Jhair Anthony Josue Choquecota Jimenez, Kendell Svante Norabuena Millones y Joe Miguel Vilca Bayona.

**Proyecto:** Detección de transacciones asociadas al lavado de activos.

**Dataset:** LI-Small_Trans.csv, IBM AML. Una fila representa una transacción sintética. Las salidas se generan con DuckDB para evitar cargar 6,9 millones de filas en pandas.

# 📦 Archivos obligatorios y simulación controlada

1. **Dataset original:** copia íntegra del archivo suministrado, sin modificaciones.
2. **Entrada simulada:** copia con problemas agregados deliberadamente para practicar limpieza. No representa la calidad original.
3. **Dataset en proceso:** datos después de las reglas de limpieza y transformaciones.
4. **Dataset final:** versión analítica. El notebook conserva el original y documenta exactamente cada modificación.

Las filas artificiales se identifican mediante `__fila_control`. Las modificaciones no deben presentarse como defectos encontrados en el archivo original.

# 0. Configuración inicial

In [1]:
import duckdb, pandas as pd, numpy as np
from pathlib import Path
from IPython.display import display
import hashlib, time
pd.set_option('display.max_columns', 100)
con = duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute("SET threads=2")
con.execute("SET enable_progress_bar=false")
con.execute("SET temp_directory='./datos/temporal_duckdb'")
def tabla(sql):
    return con.execute(sql).df()
print('Entorno listo. DuckDB:', duckdb.__version__)

Entorno listo. DuckDB: 2.0.0


# 0.1 Carga del dataset original

In [2]:
# El CSV original se conserva intacto. El notebook debe abrirse desde entrega_parcial.
ORIGINAL = Path('datos/original.csv')
PROCESO = Path('datos/proceso.csv')
FINAL = Path('datos/final.csv')
assert ORIGINAL.exists(), f'No se encuentra {ORIGINAL.resolve()}'
print('Archivo original:', ORIGINAL.resolve(), 'bytes:', ORIGINAL.stat().st_size)
print('SHA-256:', hashlib.file_digest(ORIGINAL.open('rb'), 'sha256').hexdigest())

Archivo original: /home/glender/ProyectosUNi/DataMining/entrega_parcial/datos/original.csv bytes: 650422357


SHA-256: f7a9940339c78b5d1476071505b5867c0f5319640b9c4279d13f531bf965319d


## Carga del archivo

Completa el código según sea CSV o Excel.

Debes crear:

```python
df_original
df_proceso
```

`df_proceso` debe iniciar como una copia del original.

In [3]:
# Carga de la fuente sin alterar el CSV; los identificadores se leen como texto.
con.execute("CREATE OR REPLACE TABLE original AS SELECT * FROM read_csv('datos/original.csv', all_varchar=true)")
df_original = 'original'
columnas = [r[0] for r in con.execute('DESCRIBE original').fetchall()]
q = lambda x: '"' + x.replace('"','""') + '"'
n_filas = con.execute('SELECT count(*) FROM original').fetchone()[0]
print('Original:', n_filas, 'filas y', len(columnas), 'columnas')
# 1) Monto nulo; 2) formato nulo; 3) espacios alrededor del formato;
# 4) monto negativo; 5) monto extremo artificial; 6) copia idéntica de la fila 7.
sql_simulacion = """WITH numerada AS (SELECT row_number() OVER () AS __fila_control, * FROM original),
 alterada AS (SELECT * REPLACE (
  CASE WHEN __fila_control=1 THEN NULL WHEN __fila_control=4 THEN '-1234.00'
       WHEN __fila_control=5 THEN '10000000000000000' ELSE "Amount Paid" END AS "Amount Paid",
  CASE WHEN __fila_control=2 THEN NULL WHEN __fila_control=3 THEN '  ' || "Payment Format" || '  '
       ELSE "Payment Format" END AS "Payment Format") FROM numerada)
 SELECT * FROM alterada
 UNION ALL SELECT (SELECT count(*)+1 FROM original) AS __fila_control, * EXCLUDE (__fila_control)
 FROM numerada WHERE __fila_control=7"""
con.execute('CREATE OR REPLACE TABLE entrada_simulada AS ' + sql_simulacion)
con.execute("COPY entrada_simulada TO 'datos/entrada_simulada.csv' (HEADER, DELIMITER ',')")
print('Entrada simulada:', con.execute('SELECT count(*) FROM entrada_simulada').fetchone()[0], 'filas; incluye una copia añadida.')
print('Problemas introducidos: 2 nulos, 1 categoría con espacios, 1 monto negativo, 1 monto extremo y 1 copia.')
display(tabla('SELECT __fila_control, "Amount Paid", "Payment Currency", "Payment Format" FROM entrada_simulada WHERE __fila_control<=7 ORDER BY __fila_control'))

Original: 6924049 filas y 11 columnas


Entrada simulada: 6924050 filas; incluye una copia añadida.
Problemas introducidos: 2 nulos, 1 categoría con espacios, 1 monto negativo, 1 monto extremo y 1 copia.


,__fila_control,Amount Paid,Payment Currency,Payment Format
0,1,NaN,US Dollar,Reinvestment
1,2,1858.96,US Dollar,NaN
2,3,592571.00,US Dollar,Cheque
3,4,-1234.00,US Dollar,Reinvestment
4,5,10000000000000000,US Dollar,Reinvestment
5,6,6473.62,US Dollar,Reinvestment
6,7,60562.00,US Dollar,ACH


# 0.2 Evidencia inicial — 1 punto

Muestra con código:

- Primeras 5 filas
- Dimensiones
- Tipos
- Nulos totales
- Duplicados exactos

In [4]:
display(tabla('SELECT * FROM original LIMIT 5'))
display(tabla('DESCRIBE original'))
nulos = tabla('SELECT '+', '.join('count(*) FILTER (WHERE '+q(c)+' IS NULL) AS '+q(c) for c in columnas)+' FROM original')
dup_original=con.execute('SELECT coalesce(sum(n-1),0) FROM (SELECT count(*) n FROM original GROUP BY '+', '.join(q(c) for c in columnas)+' HAVING n>1)').fetchone()[0]
print('Nulos del original:',int(nulos.iloc[0].sum()),'; duplicados exactos originales:',dup_original)
display(tabla('SELECT * EXCLUDE (__fila_control) FROM entrada_simulada LIMIT 5'))

,Timestamp,From Bank,Account,To Bank,Account_1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,011,8000ECA90,011,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,03402,80021DAD0,03402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,011,8000ECA90,001120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,03814,8006AD080,03814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,020,8006AD530,020,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


,column_name,column_type,null,key,default,extra
0,Timestamp,VARCHAR,YES,None,None,None
1,From Bank,VARCHAR,YES,None,None,None
2,Account,VARCHAR,YES,None,None,None
3,To Bank,VARCHAR,YES,None,None,None
4,Account_1,VARCHAR,YES,None,None,None
5,Amount Received,VARCHAR,YES,None,None,None
6,Receiving Currency,VARCHAR,YES,None,None,None
7,Amount Paid,VARCHAR,YES,None,None,None
8,Payment Currency,VARCHAR,YES,None,None,None
9,Payment Format,VARCHAR,YES,None,None,None


Nulos del original: 0 ; duplicados exactos originales: 8


,Timestamp,From Bank,Account,To Bank,Account_1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,011,8000ECA90,011,8000ECA90,3195403.00,US Dollar,NaN,US Dollar,Reinvestment,0
1,2022/09/01 00:21,03402,80021DAD0,03402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,NaN,0
2,2022/09/01 00:00,011,8000ECA90,001120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,03814,8006AD080,03814,8006AD080,12.32,US Dollar,-1234.00,US Dollar,Reinvestment,0
4,2022/09/01 00:00,020,8006AD530,020,8006AD530,2941.56,US Dollar,10000000000000000,US Dollar,Reinvestment,0


### Interpretación inicial
El original tiene cero nulos y 8 duplicados exactos. Para demostrar los tratamientos se creó una entrada simulada con anomalías conocidas; esa copia es distinta del original y contiene una fila adicional. No se atribuyen al origen los problemas que el equipo introdujo.

---
# 1. LIMPIEZA Y TRATAMIENTO DE CALIDAD — 5 puntos

Debes trabajar nulos, duplicados, inconsistencias, valores inválidos y outliers.

## 1.1 Valores nulos — 1.5 puntos

Debes:

1. Calcular cantidad y porcentaje de nulos
2. Seleccionar al menos una variable con nulos
3. Ddecidir eliminar, imputar o conservar
4. Ejecutar el tratamiento
5. Mostrar evidencia antes y después

In [5]:
null_sim = tabla('SELECT '+', '.join('count(*) FILTER (WHERE '+q(c)+' IS NULL) AS '+q(c) for c in columnas)+' FROM entrada_simulada')
reporte_nulos=pd.DataFrame({'variable':columnas,'original':nulos.iloc[0].values,'simulada':null_sim.iloc[0].values})
reporte_nulos['porcentaje_simulada']=100*reporte_nulos['simulada']/(n_filas+1)
display(reporte_nulos)
mediana_usd=con.execute("""SELECT quantile_cont(try_cast("Amount Paid" AS DOUBLE),0.5) FROM original WHERE "Payment Currency"='US Dollar'""").fetchone()[0]
moda=con.execute('SELECT "Payment Format" FROM original GROUP BY 1 ORDER BY count(*) DESC, 1 LIMIT 1').fetchone()[0]
print('Mediana USD usada en este ejercicio:',mediana_usd,'; moda del formato:',moda)
print('Las columnas limpias se crean en la sección 5 y se validan a continuación.')

,variable,original,simulada,porcentaje_simulada
0,Timestamp,0,0,0.000000
1,From Bank,0,0,0.000000
2,Account,0,0,0.000000
3,To Bank,0,0,0.000000
4,Account_1,0,0,0.000000
5,Amount Received,0,0,0.000000
6,Receiving Currency,0,0,0.000000
7,Amount Paid,0,1,0.000014
8,Payment Currency,0,0,0.000000
9,Payment Format,0,1,0.000014


Mediana USD usada en este ejercicio: 886.36 ; moda del formato: Cheque
Las columnas limpias se crean en la sección 5 y se validan a continuación.


### Justificación
Se agregaron artificialmente dos nulos: uno en Amount Paid y otro en Payment Format. En la copia se imputa el monto con la mediana de pagos en USD y el formato con la moda; no se alteró el original. Estos parámetros se calcularon con toda la fuente para demostrar el ejercicio; antes de entrenar un modelo deben aprenderse solo del conjunto de entrenamiento para evitar fuga.

## 1.2 Duplicados — 1 punto

Debes:

1. Detectar duplicados exactos
2. Revisar una llave lógica
3. Decidir si se eliminan o conservan
4. Demostrar el resultado

In [6]:
llave='"Timestamp", "From Bank", "Account", "To Bank", "Account_1"'
dup_sim=con.execute('SELECT coalesce(sum(n-1),0) FROM (SELECT count(*) n FROM entrada_simulada GROUP BY '+', '.join(q(c) for c in columnas)+' HAVING n>1)').fetchone()[0]
display(tabla('SELECT coalesce(sum(n-1),0) AS filas_llave_repetida, count(*) AS grupos_repetidos FROM (SELECT count(*) n FROM entrada_simulada GROUP BY '+llave+' HAVING n>1)'))
print('Duplicados exactos: original',dup_original,'; entrada simulada',dup_sim)
print('Se eliminará solo la fila artificial identificada con __fila_control =',n_filas+1)

,filas_llave_repetida,grupos_repetidos
0,136531.0,134976


Duplicados exactos: original 8 ; entrada simulada 9
Se eliminará solo la fila artificial identificada con __fila_control = 6924050


### Justificación
La simulación añadió una copia conocida de la fila 7, que puede retirarse con seguridad. Las 8 coincidencias preexistentes se conservan: la fuente no contiene identificador único que confirme que son copias. No se elimina automáticamente cada coincidencia de la llave lógica.

## 1.3 Consistencia y valores inválidos — 1.5 puntos

Selecciona al menos una variable y:

- Detecta inconsistencias o valores inválidos
- Define una regla
- Aplica la corrección
- Valida el resultado

In [7]:
def calidad(nombre):
 return tabla(f"""SELECT count(*) FILTER (WHERE "Payment Format" <> trim("Payment Format")) AS espacios_formato,
 count(*) FILTER (WHERE try_cast("Amount Paid" AS DOUBLE)<0) AS montos_negativos,
 count(*) FILTER (WHERE try_strptime("Timestamp", '%Y/%m/%d %H:%M') IS NULL) AS fechas_invalidas,
 count(*) FILTER (WHERE "Is Laundering" NOT IN ('0','1') OR "Is Laundering" IS NULL) AS etiquetas_invalidas FROM {nombre}""")
print('Calidad antes del tratamiento:');display(calidad('entrada_simulada'))
print('Regla: trim al formato; monto negativo se convierte en faltante y se imputa con mediana USD.')

Calidad antes del tratamiento:


,espacios_formato,montos_negativos,fechas_invalidas,etiquetas_invalidas
0,1,1,0,0


Regla: trim al formato; monto negativo se convierte en faltante y se imputa con mediana USD.


### Regla aplicada
Se añadió un formato con espacios y un importe negativo. Se aplica trim a la copia y el negativo se trata como inválido, sustituyéndolo por la mediana de USD. Las reglas se ejecutan dentro del pipeline y su efecto se verifica en la sección 5. La regla de no negatividad se adopta para esta simulación; una fuente real podría incluir reversos y requeriría validación de dominio.

## 1.4 Outliers — 1 punto

Selecciona al menos una variable numérica y:

1. Detecta posibles outliers
2. Utiliza IQR, boxplot u otro método justificable
3. Decide conservar, excluir, transformar o separar
4. Justifica la decisión

In [8]:
iqr = tabla("""WITH base AS (SELECT cast("Amount Paid" AS DOUBLE) monto FROM original WHERE "Payment Currency"='US Dollar'),
 q AS (SELECT quantile_cont(monto,0.25) q1,quantile_cont(monto,0.75) q3 FROM base)
 SELECT q1,q3,q3-q1 AS iqr,q1-1.5*(q3-q1) AS limite_inferior,q3+1.5*(q3-q1) AS limite_superior,
 (SELECT count(*) FROM base WHERE monto>q.q3+1.5*(q.q3-q.q1) OR monto<q.q1-1.5*(q.q3-q.q1)) AS posibles_atipicos FROM q""")
limite_usd=float(iqr['limite_superior'].iloc[0]);display(iqr)
print('Valor artificial extremo antes:',con.execute('SELECT "Amount Paid" FROM entrada_simulada WHERE __fila_control=5').fetchone()[0])
print('Valor de reemplazo por límite IQR en USD:',limite_usd)

,q1,q3,iqr,limite_inferior,limite_superior,posibles_atipicos
0,157.57,5202.9,5045.33,-7410.425,12770.895,428762


Valor artificial extremo antes: 10000000000000000
Valor de reemplazo por límite IQR en USD: 12770.895


### Decisión sobre los outliers
El valor de la fila 5 fue colocado artificialmente y se limita al umbral superior IQR de pagos en USD. Los demás valores atípicos se conservan: el equipo no dispone de evidencia de error para eliminarlos o truncarlos. Los límites se calcularon sobre toda la fuente para esta demostración; en modelado deben calcularse solo con entrenamiento.

---
# 2. TRANSFORMACIÓN Y CODIFICACIÓN — 4 puntos

## 2.1 Conversión de tipos — 1 punto

Identifica al menos una variable cuyo tipo técnico no represente correctamente su significado.

Ejemplos:

- Número guardado como `object`;
- Fecha almacenada como texto.

Debes mostrar el tipo antes y después.

In [9]:
display(tabla('DESCRIBE SELECT "Timestamp", "Amount Paid", "From Bank" FROM entrada_simulada'))
display(tabla("""SELECT "Timestamp" fecha_texto, try_strptime("Timestamp", '%Y/%m/%d %H:%M') fecha_convertida, typeof(try_strptime("Timestamp", '%Y/%m/%d %H:%M')) tipo_fecha, "From Bank" banco_texto FROM entrada_simulada LIMIT 5"""))
print('From Bank permanece VARCHAR: identifica un banco, no es una magnitud.')

,column_name,column_type,null,key,default,extra
0,Timestamp,VARCHAR,YES,None,None,None
1,Amount Paid,VARCHAR,YES,None,None,None
2,From Bank,VARCHAR,YES,None,None,None


,fecha_texto,fecha_convertida,tipo_fecha,banco_texto
0,2022/09/01 00:08,2022-09-01 00:08:00,TIMESTAMP,011
1,2022/09/01 00:21,2022-09-01 00:21:00,TIMESTAMP,03402
2,2022/09/01 00:00,2022-09-01 00:00:00,TIMESTAMP,011
3,2022/09/01 00:16,2022-09-01 00:16:00,TIMESTAMP,03814
4,2022/09/01 00:00,2022-09-01 00:00:00,TIMESTAMP,020


From Bank permanece VARCHAR: identifica un banco, no es una magnitud.


## 2.2 Codificación categórica — 1.5 puntos

Aplica una técnica adecuada:

- Binaria
- Ordinal
- One-Hot Encoding

Debes justificar la técnica elegida.

In [10]:
display(tabla('SELECT DISTINCT trim("Payment Format") AS formato FROM entrada_simulada ORDER BY 1'))
display(tabla("""SELECT trim("Payment Format") AS formato, (trim("Payment Format")='Cheque')::UTINYINT AS formato_cheque FROM entrada_simulada LIMIT 10"""))

,formato
0,ACH
1,Bitcoin
2,Cash
3,Cheque
4,Credit Card
5,Reinvestment
6,Wire
7,NaN


,formato,formato_cheque
0,Reinvestment,0
1,NaN,<NA>
2,Cheque,1
3,Reinvestment,0
4,Reinvestment,0
5,Reinvestment,0
6,ACH,0
7,Reinvestment,0
8,Credit Card,0
9,Reinvestment,0


### Justificación

**Respuesta:**

El formato de pago es nominal; se crean indicadores one-hot para las siete categorías observadas, sin imponer orden. La moneda se conserva como categoría en texto y deberá codificarse dentro del pipeline del modelo posterior.

## 2.3 Escalamiento o discretización — 1.5 puntos

Aplica al menos una:

- Min-Max
- StandardScaler
- Discretización / binning

Muestra evidencia antes y después.

In [11]:
display(tabla("""SELECT "Amount Paid" monto_original,"Payment Currency" moneda,
 CASE WHEN cast("Amount Paid" AS DOUBLE)<1000 THEN 'menor_1000'
      WHEN cast("Amount Paid" AS DOUBLE)<10000 THEN '1000_a_9999' ELSE '10000_o_mas' END AS tramo
 FROM entrada_simulada LIMIT 10"""))

,monto_original,moneda,tramo
0,NaN,US Dollar,10000_o_mas
1,1858.96,US Dollar,1000_a_9999
2,592571.00,US Dollar,10000_o_mas
3,-1234.00,US Dollar,menor_1000
4,10000000000000000,US Dollar,10000_o_mas
5,6473.62,US Dollar,1000_a_9999
6,60562.00,US Dollar,10000_o_mas
7,22.97,US Dollar,menor_1000
8,43.53,US Dollar,menor_1000
9,5.04,US Dollar,menor_1000


### Justificación

**Respuesta:**

La discretización usa cortes fijos de 1.000 y 10.000 unidades de la moneda de pago. Son cortes descriptivos, no umbrales oficiales de riesgo ni montos comparables entre divisas. Se conserva también el importe numérico y su moneda; no se ajusta un escalador sobre todo el dataset.

---
# 3. INGENIERÍA DE CARACTERÍSTICAS — 3 puntos

Crea al menos **2 nuevas variables** útiles para el proyecto.

Cada una debe tener:

- Variables de origen
- Fórmula o lógica
- Código
- Interpretación
- Utilidad

## Variable nueva 1

In [12]:
display(tabla("""SELECT "Timestamp", extract(hour FROM try_strptime("Timestamp", '%Y/%m/%d %H:%M')) AS hora FROM entrada_simulada LIMIT 10"""))

,Timestamp,hora
0,2022/09/01 00:08,0
1,2022/09/01 00:21,0
2,2022/09/01 00:00,0
3,2022/09/01 00:16,0
4,2022/09/01 00:00,0
5,2022/09/01 00:24,0
6,2022/09/01 00:17,0
7,2022/09/01 00:07,0
8,2022/09/01 00:28,0
9,2022/09/01 00:22,0


**Origen / fórmula:**  
**Interpretación:**  
**Utilidad:**

**Origen / fórmula:** hora = hora extraída de Timestamp. **Interpretación:** entero entre 0 y 23. **Utilidad:** estudiar patrones horarios de transacciones.

## Variable nueva 2

In [13]:
display(tabla("""SELECT "Timestamp", extract(isodow FROM try_strptime("Timestamp", '%Y/%m/%d %H:%M')) AS dia_semana FROM entrada_simulada LIMIT 10"""))

,Timestamp,dia_semana
0,2022/09/01 00:08,4
1,2022/09/01 00:21,4
2,2022/09/01 00:00,4
3,2022/09/01 00:16,4
4,2022/09/01 00:00,4
5,2022/09/01 00:24,4
6,2022/09/01 00:17,4
7,2022/09/01 00:07,4
8,2022/09/01 00:28,4
9,2022/09/01 00:22,4


**Origen / fórmula:**  
**Interpretación:**  
**Utilidad:**

**Origen / fórmula:** día de la semana ISO extraído de Timestamp (1 lunes, 7 domingo). **Utilidad:** estudiar diferencias temporales entre operaciones.

---
# 4. SELECCIÓN O REDUCCIÓN DE VARIABLES — 2 puntos

Debes identificar:

- Todas variables que conservarás
- Las variable que excluirás
- Las variable que transformarás o reemplazarás por una derivada.

In [14]:
conservar = ['monto_pagado','moneda_pago','tramo_monto','hora','dia_semana','formato_ach','formato_bitcoin','formato_cash','formato_cheque','formato_credit_card','formato_reinvestment','formato_wire','etiqueta']
excluir = ['Timestamp','From Bank','Account','To Bank','Account_1','Amount Received','Receiving Currency','Amount Paid','Payment Currency','Payment Format','Is Laundering']
print('Final:', conservar)
print('Columnas originales sustituidas o excluidas:', excluir)

Final: ['monto_pagado', 'moneda_pago', 'tramo_monto', 'hora', 'dia_semana', 'formato_ach', 'formato_bitcoin', 'formato_cash', 'formato_cheque', 'formato_credit_card', 'formato_reinvestment', 'formato_wire', 'etiqueta']
Columnas originales sustituidas o excluidas: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account_1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


## Revisión de Data Leakage
La etiqueta se conserva solo como objetivo. Se excluyen identificadores y campos de recepción por cautela temporal. Las estadísticas de imputación y el IQR usados aquí emplean todo el original como ejercicio demostrativo: antes de modelar, se deben calcular de nuevo únicamente con el conjunto de entrenamiento, después de la separación temporal. La disponibilidad de cada predictor requiere confirmación.

---
# 5. DATASET EN PROCESO — 1 punto

En este punto debes guardar una versión intermedia.

Debe evidenciar las transformaciones realizadas antes de construir el dataset final.

In [15]:
def preparar_datos(nombre_tabla):
    """Limpia una entrada simulada documentada y construye el dataset analítico."""
    if nombre_tabla != 'entrada_simulada': raise ValueError('Usar la copia simulada identificada')
    moda_sql = moda.replace("'", "''")
    intermedio = f"""WITH limpia AS (
      SELECT * EXCLUDE (__fila_control) REPLACE (
        CASE WHEN "Amount Paid" IS NULL OR try_cast("Amount Paid" AS DOUBLE)<0 THEN '{mediana_usd}'
             WHEN __fila_control=5 THEN '{limite_usd}' ELSE "Amount Paid" END AS "Amount Paid",
        CASE WHEN "Payment Format" IS NULL THEN '{moda_sql}' ELSE trim("Payment Format") END AS "Payment Format")
      FROM {nombre_tabla} WHERE __fila_control <= {n_filas})
    SELECT *, try_strptime("Timestamp", '%Y/%m/%d %H:%M') AS fecha_convertida,
      try_cast("Amount Paid" AS DOUBLE) AS monto_pagado, trim("Payment Currency") AS moneda_pago,
      trim("Payment Format") AS formato_limpio, try_cast("Is Laundering" AS UTINYINT) AS etiqueta,
      extract(hour FROM try_strptime("Timestamp", '%Y/%m/%d %H:%M'))::UTINYINT AS hora,
      extract(isodow FROM try_strptime("Timestamp", '%Y/%m/%d %H:%M'))::UTINYINT AS dia_semana,
      CASE WHEN try_cast("Amount Paid" AS DOUBLE)<1000 THEN 'menor_1000' WHEN try_cast("Amount Paid" AS DOUBLE)<10000 THEN '1000_a_9999' ELSE '10000_o_mas' END AS tramo_monto,
      (trim("Payment Format")='ACH')::UTINYINT formato_ach, (trim("Payment Format")='Bitcoin')::UTINYINT formato_bitcoin,
      (trim("Payment Format")='Cash')::UTINYINT formato_cash, (trim("Payment Format")='Cheque')::UTINYINT formato_cheque,
      (trim("Payment Format")='Credit Card')::UTINYINT formato_credit_card, (trim("Payment Format")='Reinvestment')::UTINYINT formato_reinvestment,
      (trim("Payment Format")='Wire')::UTINYINT formato_wire FROM limpia"""
    final='SELECT '+', '.join(conservar)+' FROM ('+intermedio+')'
    return intermedio,final
sql_proceso,sql_final=preparar_datos('entrada_simulada')
con.execute('CREATE OR REPLACE TABLE proceso AS '+sql_proceso)
con.execute("COPY proceso TO 'datos/proceso.csv' (HEADER, DELIMITER ',')")
print('Guardado:',PROCESO,'bytes:',PROCESO.stat().st_size)

Guardado: datos/proceso.csv bytes: 1181151085


## Evidencia del dataset en proceso

Muestra:

- Dimensiones
- Primeras filas
- Nulos
- Duplicados
- Tipos

In [16]:
print('Proceso:',con.execute('SELECT count(*) FROM proceso').fetchone()[0],'filas;',len(con.execute('DESCRIBE proceso').fetchall()),'columnas')
display(tabla('SELECT * FROM proceso LIMIT 5'))
print('Calidad después de la limpieza:');display(calidad('proceso'))
null_proceso=tabla('SELECT '+', '.join('count(*) FILTER (WHERE '+q(c)+' IS NULL) AS '+q(c) for c in [r[0] for r in con.execute('DESCRIBE proceso').fetchall()])+' FROM proceso')
print('Nulos totales antes:',int(null_sim.iloc[0].sum()),'; después:',int(null_proceso.iloc[0].sum()))
dup_proceso=con.execute('SELECT coalesce(sum(n-1),0) FROM (SELECT count(*) n FROM proceso GROUP BY '+', '.join(q(c) for c in columnas)+' HAVING n>1)').fetchone()[0]
print('Duplicados exactos antes:',dup_sim,'; después:',dup_proceso)
print('Casos de control después:');display(tabla('SELECT "Amount Paid", "Payment Currency", "Payment Format" FROM proceso LIMIT 5'))
display(tabla('DESCRIBE proceso'))

Proceso: 6924049 filas; 26 columnas


,Timestamp,From Bank,Account,To Bank,Account_1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,fecha_convertida,monto_pagado,moneda_pago,formato_limpio,etiqueta,hora,dia_semana,tramo_monto,formato_ach,formato_bitcoin,formato_cash,formato_cheque,formato_credit_card,formato_reinvestment,formato_wire
0,2022/09/01 00:08,011,8000ECA90,011,8000ECA90,3195403.00,US Dollar,886.36,US Dollar,Reinvestment,0,2022-09-01 00:08:00,886.360,US Dollar,Reinvestment,0,0,4,menor_1000,0,0,0,0,0,1,0
1,2022/09/01 00:21,03402,80021DAD0,03402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Cheque,0,2022-09-01 00:21:00,1858.960,US Dollar,Cheque,0,0,4,1000_a_9999,0,0,0,1,0,0,0
2,2022/09/01 00:00,011,8000ECA90,001120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0,2022-09-01 00:00:00,592571.000,US Dollar,Cheque,0,0,4,10000_o_mas,0,0,0,1,0,0,0
3,2022/09/01 00:16,03814,8006AD080,03814,8006AD080,12.32,US Dollar,886.36,US Dollar,Reinvestment,0,2022-09-01 00:16:00,886.360,US Dollar,Reinvestment,0,0,4,menor_1000,0,0,0,0,0,1,0
4,2022/09/01 00:00,020,8006AD530,020,8006AD530,2941.56,US Dollar,12770.895,US Dollar,Reinvestment,0,2022-09-01 00:00:00,12770.895,US Dollar,Reinvestment,0,0,4,10000_o_mas,0,0,0,0,0,1,0


Calidad después de la limpieza:


,espacios_formato,montos_negativos,fechas_invalidas,etiquetas_invalidas
0,0,0,0,0


Nulos totales antes: 2 ; después: 0


Duplicados exactos antes: 9 ; después: 8
Casos de control después:


,Amount Paid,Payment Currency,Payment Format
0,886.36,US Dollar,Reinvestment
1,1858.96,US Dollar,Cheque
2,592571.00,US Dollar,Cheque
3,886.36,US Dollar,Reinvestment
4,12770.895,US Dollar,Reinvestment


,column_name,column_type,null,key,default,extra
0,Timestamp,VARCHAR,YES,None,None,None
1,From Bank,VARCHAR,YES,None,None,None
2,Account,VARCHAR,YES,None,None,None
3,To Bank,VARCHAR,YES,None,None,None
4,Account_1,VARCHAR,YES,None,None,None
5,Amount Received,VARCHAR,YES,None,None,None
6,Receiving Currency,VARCHAR,YES,None,None,None
7,Amount Paid,VARCHAR,YES,None,None,None
8,Payment Currency,VARCHAR,YES,None,None,None
9,Payment Format,VARCHAR,YES,None,None,None


---
# 6. PIPELINE REPRODUCIBLE — 2 puntos

Organiza tu proceso en una función para que pueda ejecutarse nuevamente desde el dataset original.

In [17]:
# La función definida en la sección anterior se ejecuta de nuevo desde la entrada simulada.
print(preparar_datos('entrada_simulada')[1][:250]+'...')

SELECT monto_pagado, moneda_pago, tramo_monto, hora, dia_semana, formato_ach, formato_bitcoin, formato_cash, formato_cheque, formato_credit_card, formato_reinvestment, formato_wire, etiqueta FROM (WITH limpia AS (
      SELECT * EXCLUDE (__fila_contr...


### Ejecuta el pipeline
Debe producir el mismo resultado al recrear la entrada simulada desde el original. Se verifica con una comparación exacta de las filas.

In [18]:
# Reconstrucción completa desde el original, incluida la simulación controlada.
con.execute('CREATE OR REPLACE TABLE entrada_reproducida AS '+sql_simulacion)
# La función admite solo el nombre fijado de la entrada; se conserva su control de seguridad.
_, sql_reproducible=preparar_datos('entrada_simulada')
sql_reproducible=sql_reproducible.replace('FROM entrada_simulada WHERE','FROM entrada_reproducida WHERE')
con.execute('CREATE OR REPLACE TABLE final AS '+sql_reproducible)
print('Pipeline desde el original y la simulación:',con.execute('SELECT count(*) FROM final').fetchone()[0],'filas')
cols=', '.join(conservar)
comparacion_exacta=con.execute(f"SELECT count(*) FROM ((SELECT {cols} FROM proceso EXCEPT ALL SELECT {cols} FROM final) UNION ALL (SELECT {cols} FROM final EXCEPT ALL SELECT {cols} FROM proceso))").fetchone()[0]
print('Diferencias de filas entre proceso seleccionado y pipeline:',comparacion_exacta)
assert comparacion_exacta==0

Pipeline desde el original y la simulación: 6924049 filas


Diferencias de filas entre proceso seleccionado y pipeline: 0


---
# 7. DATASET FINAL PREPARADO — 2 puntos

## 7.1 Construcción del dataset final

In [19]:
df_final = 'final'
display(tabla('SELECT * FROM final LIMIT 5'))

,monto_pagado,moneda_pago,tramo_monto,hora,dia_semana,formato_ach,formato_bitcoin,formato_cash,formato_cheque,formato_credit_card,formato_reinvestment,formato_wire,etiqueta
0,886.360,US Dollar,menor_1000,0,4,0,0,0,0,0,1,0,0
1,1858.960,US Dollar,1000_a_9999,0,4,0,0,0,1,0,0,0,0
2,592571.000,US Dollar,10000_o_mas,0,4,0,0,0,1,0,0,0,0
3,886.360,US Dollar,menor_1000,0,4,0,0,0,0,0,1,0,0
4,12770.895,US Dollar,10000_o_mas,0,4,0,0,0,0,0,1,0,0


## 7.2 Validación final

Debes mostrar:

- Dimensiones
- Nulos
- Duplicados
- Tipos
- Columnas finales

In [20]:
final_cols = [r[0] for r in con.execute('DESCRIBE final').fetchall()]
null_final = tabla('SELECT ' + ', '.join('count(*) FILTER (WHERE '+q(c)+' IS NULL) AS '+q(c) for c in final_cols) + ' FROM final')
display(tabla('DESCRIBE final'))
display(null_final.T.rename(columns={0:'nulos'}))
dup_final = con.execute('SELECT coalesce(sum(n-1),0) FROM (SELECT count(*) n FROM final GROUP BY ' + ', '.join(q(c) for c in final_cols) + ' HAVING n>1)').fetchone()[0]
print('Duplicados exactos en variables finales:',dup_final)
print('Distribución de etiqueta:'); display(tabla('SELECT etiqueta, count(*) n, round(100.0*count(*)/sum(count(*)) over (),5) porcentaje FROM final GROUP BY 1 ORDER BY 1'))

,column_name,column_type,null,key,default,extra
0,monto_pagado,DOUBLE,YES,None,None,None
1,moneda_pago,VARCHAR,YES,None,None,None
2,tramo_monto,VARCHAR,YES,None,None,None
3,hora,UTINYINT,YES,None,None,None
4,dia_semana,UTINYINT,YES,None,None,None
5,formato_ach,UTINYINT,YES,None,None,None
6,formato_bitcoin,UTINYINT,YES,None,None,None
7,formato_cash,UTINYINT,YES,None,None,None
8,formato_cheque,UTINYINT,YES,None,None,None
9,formato_credit_card,UTINYINT,YES,None,None,None


,nulos
monto_pagado,0
moneda_pago,0
tramo_monto,0
hora,0
dia_semana,0
formato_ach,0
formato_bitcoin,0
formato_cash,0
formato_cheque,0
formato_credit_card,0


Duplicados exactos en variables finales: 284698
Distribución de etiqueta:


,etiqueta,n,porcentaje
0,0,6920484,99.94851
1,1,3565,0.05149


## 7.3 Comparación obligatoria

Compara:

- Original
- Proceso
- Final

In [21]:
comparacion=pd.DataFrame([
 {'Dataset':'Original','Filas':n_filas,'Columnas':len(columnas),'Nulos':int(nulos.iloc[0].sum()),'Duplicados':dup_original},
 {'Dataset':'Entrada simulada','Filas':n_filas+1,'Columnas':len(columnas)+1,'Nulos':int(null_sim.iloc[0].sum()),'Duplicados':dup_sim},
 {'Dataset':'Proceso','Filas':con.execute('SELECT count(*) FROM proceso').fetchone()[0],'Columnas':len(con.execute('DESCRIBE proceso').fetchall()),'Nulos':int(null_proceso.iloc[0].sum()),'Duplicados':dup_proceso},
 {'Dataset':'Final','Filas':con.execute('SELECT count(*) FROM final').fetchone()[0],'Columnas':len(final_cols),'Nulos':int(null_final.iloc[0].sum()),'Duplicados':dup_final}])
display(comparacion)
print('La entrada simulada incluye __fila_control; se excluye de proceso y final.')
print('Duplicados finales: coincidencias en menos columnas, no se eliminan filas originales.')

,Dataset,Filas,Columnas,Nulos,Duplicados
0,Original,6924049,11,0,8
1,Entrada simulada,6924050,12,2,9
2,Proceso,6924049,26,0,8
3,Final,6924049,13,0,284698


La entrada simulada incluye __fila_control; se excluye de proceso y final.
Duplicados finales: coincidencias en menos columnas, no se eliminan filas originales.


## 7.4 Guarda el dataset final

In [22]:
con.execute("COPY final TO 'datos/final.csv' (HEADER, DELIMITER ',')")
print('Guardado:',FINAL,'bytes:',FINAL.stat().st_size)
print('Original SHA-256 tras procesamiento:',hashlib.file_digest(ORIGINAL.open('rb'),'sha256').hexdigest())

Guardado: datos/final.csv bytes: 334823736


Original SHA-256 tras procesamiento: f7a9940339c78b5d1476071505b5867c0f5319640b9c4279d13f531bf965319d


## 8. Conclusión técnica
La fuente original no tiene nulos y presenta 8 duplicados exactos que se conservaron. Para practicar se introdujeron en una copia dos nulos, un formato con espacios, un monto negativo, un extremo y una fila duplicada; se imputó, normalizó, limitó el extremo conocido y retiró solo la copia añadida. Se derivaron hora y día de semana, tramos y siete indicadores. Se excluyeron identificadores y campos de recepción por cautela temporal. El resultado permite iniciar exploración de modelado; los parámetros deben reajustarse solo con datos de entrenamiento y los datos sintéticos no demuestran desempeño real.

# ✅ CHECKLIST DE ENTREGA
- [x] Notebook ejecutado con resultados
- [x] Dataset original intacto
- [x] Entrada simulada identificada y reproducible
- [x] Dataset en proceso y dataset final
- [x] Evidencia antes y después y justificaciones
- [x] Pipeline reproducible y comparación

**Transparencia:** los problemas agregados son simulados. Los parámetros estimados con el conjunto completo son solo para el ejercicio; deben reajustarse en entrenamiento antes de modelar.